In [ ]:
! pip install --upgrade langchain langchain-openai langchain-core langchain_community docx2txt pypdf langchain_chroma sentence_transformers

In [ ]:

import langchain
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


from langchain_openai import ChatOpenAI, OpenAIEmbeddings


from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter


from langchain_chroma import Chroma


import docx2txt
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer


In [ ]:
import langchain
print(langchain.__version__)

In [ ]:
import os

os.environ["OPENAI_API_KEY"]= "your_openai_api_key_here"
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_API_KEY"] = "your_langchain_api_key_here"
os.environ["LANGCHAIN_PROJECT"] = "RAG PROJECT"


Calling the llm


In [ ]:
import langchain_openai
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model= " ")
llm_response = llm.invoke("") #pass your prompt to check whether it is working or not!!!

llm_response

Parsing O/P

In [ ]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()
output_parser.invoke(llm_response)

Simple chain


In [ ]:
chain =  llm | output_parser
chain.invoke(" ")


Structured 0utput

In [ ]:
from typing import List
from pydantic import BaseModel, Field

class MobileReview(BaseModel):
        
    phone_model: str = Field(description="Name and model of the phone")
    rating: float = Field(description="Overall rating out of 5")
    pros: List[str] = Field(description="List of positive aspects")
    cons: List[str] = Field(description="List of negative aspects")
    summary: str = Field(description="Brief summary of the review")

review_text = """
Just got my hands on the new Galaxy S21 and wow, this thing is slick! The screen is gorgeous,
colors pop like crazy. Camera's insane too, especially at night - my Insta game's never been
stronger. Battery life's solid, lasts me all day no problem.
Not gonna lie though, it's pretty pricey. And what's with ditching the charger? C'mon Samsung.
Also, still getting used to the new button layout, keep hitting Bixby by mistake.
Overall, I'd say it's a solid 4 out of 5. Great phone, but a few annoying quirks keep it from
being perfect. If you're due for an upgrade, definitely worth checking out!
"""

structured_llm = llm.with_structured_output(MobileReview)
output = structured_llm.invoke(review_text)
print(output)


Prompt template

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
prompt  = ChatPromptTemplate.from_template(" ")
prompt.invoke({" ":" "})


In [ ]:
chain = prompt | llm | output_parser
chain.invoke({" " : " "})

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

#define the prompt
prompt = ChatPromptTemplate.from_template("")

#initialize the LLM
llm = ChatOpenAI(model=" ")

#define the o/p parser[]
output_parser= StrOutputParser()

#compose the chain
chain = prompt | llm | output_parser

#usinf the chain
result = chain.invoke({" ":" "})
print(result)

LLM Messages

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage

system_message = SystemMessage(content=" ")
human_message = HumanMessage(content=" ")
llm.invoke([system_message, human_message])

In [ ]:
template = ChatPromptTemplate([
    ("system", " "),
    ("human", " ")
])

prompt_value = template.invoke(
    {
        "topic": " "
    }
)

prompt_value

In [ ]:
llm.invoke(prompt_value)

In [ ]:
! pip install docx2txt pypdf unstructured

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from typing import List
from langchain_core.documents import Document


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap=200,
    length_function = len
)

docx_loader = Docx2txtLoader(" ")  
documents = docx_loader.load()

splits = text_splitter.split_documents(documents)

print(f"Split the documents into {len(splits)} chunks.")



In [ ]:
#funtion to load documents from a folder

def load_documents(folder_path: str) -> List[Document]:
    documents = []
    for filename in os.listdir(folder_path):
        file_path= os.path.join(folder_path, filename)

        if filename.endswith('.pdf'):
            loader = PyPDFLoader(file_path)
        elif filename.endswith('.docx'):
            loader = Docx2txtLoader(file_path)
        else:
            print(f"Unsupported file type: {filename}")
            continue
        documents.extend(loader.load())
    return documents

#load documents from a folder 

folder_path = " "

documents = load_documents(folder_path)

print(f"Loaded {len(documents)} documents from the folder.")
splits= text_splitter.split_documents(documents)
print(f"Split the documents into {len(splits)} chunks.")



In [ ]:
embeddings = OpenAIEmbeddings()

###Embedding Documents

documents_embeddings= embeddings.embed_documents([split.page_content for split in splits])

print(f"Created Embeddings for {len(documents_embeddings)} document chunks.")

documents_embeddings[0]


In [ ]:
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings

embeddings_function = SentenceTransformerEmbeddings(model_name = "all-MiniLM-L6-v2")
document_embeddings = embeddings_function.embed_documents([split.page.content for split in splits])
documents_embeddings[0]

In [ ]:
! pip install langchain_chroma -q

In [ ]:
#import shutil
##shutil.rmtree('chroma_db')

Create and persist Chroma Vector Store

In [ ]:
from langchain_chroma import Chroma

embeddings_function = OpenAIEmbeddings()
collection_name = "my_collection"

vectorstore = Chroma.from_documents(collection_name=collection_name, documents=splits, embedding=embeddings_function)
#db.persist()

print("Vector Store created and persisted to './chroma_db'")


In [ ]:
# performing similarity search

query = " "
search_results = vectorstore.similarity_search(query,k=2)

print(f"\nTop 2 most relevant chunks for the query: '{query}'\n")
for i, result in enumerate(search_results,1):
    print(f"Result {i}:")
    print(f"Source: {result.metadata.get('souurce','Unknown')}")

    print(f"Content: {result.page_content}")
    print()

In [ ]:
retriever =  vectorstore.as_retriever(search_kwargs={"k":2})

retriever.invoke("") ##put your query/prompt here


In [ ]:
from langchain_core.prompts import ChatMessagePromptTemplate
template = """Answer the question based only on the following context:
{context}

Questin:{question}

Answer: """

prompt = ChatMessagePromptTemplate.from_template(template)


In [ ]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()} | prompt
)
rag_chain.invoke("   ")   ###put the same query here



In [ ]:
def doc2str(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
rag_chain= (
    {"context": retriever | doc2str, "question": RunnablePassthrough()} | prompt
)
rag_chain.invoke(" ")

  

In [ ]:
rag_chain= (
    {"context": retriever | doc2str, "question": RunnablePassthrough()} 
    | prompt
    | llm 
    | StrOutputParser()

)

question = " "
response = rag_chain.invoke(question)
print(response)

<h1> Conversational RAG </h1>

<h5> HAndling Follow up Questions



In [ ]:
# Example conversation

from langchain_core.messages import HumanMessage, AIMessage

chat_history = []
chat_history.extend([
    HumanMessage(content=question),
    AIMessage(content=response)
])

In [ ]:
chat_history

In [ ]:
from langchain_core.prompts import MessagesPlaceholder

contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return as it is."

)
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

#history_aware_retriever = create_history_aware_retriever(
#   llm, retriever, contextualize_q_prompt
#  )

contextualize_chain = contextualize_q_prompt | llm | StrOutputParser
contextualize_chain.invoke({"input": "     ", "chat_history": chat_history})

In [ ]:
from langchain_classic.chains import create_history_aware_retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)
history_aware_retriever.invoke({"input": "  ","chat_history": chat_history})

In [ ]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI Assistant. Use the following context to answer the user's question."),
    # ("system", "Tell me a joke on Programming"),
    ("system", "Context: {context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human","{input}")
])

question_answer_chain = create_stuff_documents_chain(llm , qa_prompt)

rag_chain= create_retrieval_chain(history_aware_retriever, question_answer_chain)


In [ ]:
rag_chain.invoke({"input": "  ","chat_history":chat_history})

<h1> Multi User Chatbot

In [ ]:
import sqlite3
from datetime import datetime

DB_NAME = "rag_app.db"

def get_db_connection():
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row
    return conn

def create_application_logs():
    conn = get_db_connection()
    conn.execute('''CREATE TABLE IF NOT EXISTS application_logs
                    (id INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT,
                    user_query TEXT,
                    gpt_response TEXT,
                    model TEXT,
                    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)''')
    conn.close()

def insert_application_logs(session_id, user_query, gpt_response, model):
     conn = get_db_connection()
     conn.execute('INSERT INTO application_logs (session_id, user_query, gpt_response, model) VALUES (?,?,?,?)',
                  (session_id, user_query, gpt_response, model))
     conn.commit()
     conn.close()

def get_chat_history(session_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT user_query,  gpt_response FROM application_logs WHERE session_id = ? ORDER BY created_at',(session_id))
    messages = []
    for row in cursor.fetchall():
        messages.extend([
            {"role": "human", "content": row['user_query']},
            {"role": "ai", "content": row['gpt_response']}
        ])
    conn.close()
    return messages

# Initialize the database
create_application_logs()


In [ ]:
import uuid
session_id = str(uuid.uuid4())

chat_history = get_chat_history(session_id)

print(chat_history)

question1= "  "
answer1= rag_chain.invoke({"input": question1, "chat_history":chat_history})['answer']

insert_application_logs(session_id, question1, answer1 ,"gpt-4-o-mini")
print(f"Human: {question1}")
print(f"AI: {answer1}\n")

In [ ]:
question2 = "  "
chat_history = get_chat_history(session_id)

print(chat_history)

answer2 = rag_chain.invoke({"input": question2, "chat_history":chat_history})['answer']
insert_application_logs(session_id, question2, answer2, "gpt-3.5-turbo")
print(f"Human: {question2}")
print(f"AI: {answer2}\n")